# 06 · CO4: Generative AI / LLM application & prompt engineering (8 marks)
A **local LLM** (Qwen2.5-1.5B-Instruct, served by Ollama on `localhost`, no external API) is integrated with the
trained NLP model:
1. **LLM-as-judge:** four prompt strategies (zero-shot, few-shot, chain-of-thought, and an improved v2 "score-first" prompt) compared with the trained model
2. **Grounded explanation:** the LLM explains a prediction using the NLP model's measured signals
3. **Controlled rewriting:** make a question easier/harder, then **verify** with the trained model
4. **Question generation** at a target difficulty, verified the same way

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option('display.max_colwidth', 120)
REPORTS = ROOT / 'reports'; FIG = REPORTS / 'figures'
def show(name): display(Image(filename=str(FIG / name)))
def rj(name): return json.loads((REPORTS / name).read_text())

In [2]:
from src.llm import genai
print('Local LLM available:', genai.is_available(), '| model:', genai.MODEL)
for p in sorted((ROOT/'prompts').glob('*.txt')): print(f'\n===== {p.name} =====\n' + p.read_text())

Local LLM available: True | model: qwen2.5:1.5b

===== explain.txt =====
A trained NLP model rated this exam question as {level} (difficulty score {score}/100).

Question: """{question}"""

Signals measured by the model:
{signals}

In 3-4 short bullet points, explain to a teacher WHY this question has this difficulty. Mention concrete aspects of the question (concepts, steps, wording). Do not repeat the numbers mechanically and do not answer the question.


===== generate.txt =====
Write {n} new exam questions about the topic "{topic}" at {level} difficulty.

{level} means: {definition}

Output ONLY a JSON object: {{"questions": ["question 1", "question 2", ...]}}


===== rate_chain_of_thought.txt =====
Rate the difficulty of the following exam question. Think step by step before deciding.

Levels: Easy (0-33) = basic recall, one step, school level. Medium (34-66) = apply/understand a concept, a few steps, high-school level. Hard (67-100) = multi-step reasoning or advanced/specialised 

## 1 · Prompt-strategy evaluation (90 balanced test questions: 30 Easy / 30 Medium / 30 Hard)

In [3]:
le = rj('llm_prompt_eval.json')
pd.DataFrame({k:v for k,v in le.items() if k!='rewrite'}).T[['accuracy','macro_f1','qwk','mae','spearman','accuracy_level_from_score','seconds_per_question','predicted_level_distribution']]

,accuracy,macro_f1,qwk,mae,spearman,accuracy_level_from_score,seconds_per_question,predicted_level_distribution
NLP ensemble (trained),0.7444,0.742,0.7869,8.3089,0.8509,NaN,NaN,NaN
LLM score_first,0.4111,0.3851,0.2273,21.9222,0.158,0.4111,2.44,"{'Medium': 62, 'Hard': 16, 'Easy': 12}"
LLM zero_shot,0.3333,0.1667,0.0,21.2444,0.3063,0.4222,1.18,{'Easy': 90}
LLM few_shot,0.3889,0.4073,0.3214,25.3778,0.0605,0.3444,1.36,"{'Easy': 38, 'Medium': 38, 'Hard': 14}"
LLM chain_of_thought,0.3333,0.1667,0.0,24.3667,0.1234,0.3333,5.11,{'Easy': 90}


In [4]:
lp = pd.read_csv(REPORTS/'llm_prompt_predictions.csv')
lp[['question','level','nlp_level','llm_zero_shot_level','llm_few_shot_level','llm_chain_of_thought_level','llm_score_first_level']].head(12)

,question,level,nlp_level,llm_zero_shot_level,llm_few_shot_level,llm_chain_of_thought_level,llm_score_first_level
0,Running when you're out of shape can easily lead to something that might be distressing and uncomfortable. What can ...,Easy,Medium,Easy,Easy,Easy,Medium
1,What state has a county called columbia and is the furthest in the direction that the sun sets?,Easy,Easy,Easy,Easy,Easy,Medium
2,According to the passage you usually do everything on the first day of school except _ .,Easy,Easy,Easy,Easy,Easy,Medium
3,The visible light spectrum can be subdivided according to,Easy,Medium,Easy,Easy,Easy,Medium
4,"A glass of ice water is placed on a table. After 10 minutes, there are drops of water on the outside surface of the ...",Easy,Medium,Easy,Medium,Easy,Medium
5,"Coal, oil, and natural gas have high potential energies, which makes them valuable energy resources. In order to ben...",Easy,Easy,Easy,Medium,Easy,Medium
6,"At a gym, the blue weights are 2 pounds each, and the green weights are 3 pounds each. Harry put 4 blue weights and ...",Easy,Easy,Easy,Medium,Easy,Medium
7,What helps pumice float in water?,Easy,Easy,Easy,Easy,Easy,Medium
8,"A water tank will hold 50 gallons. What flow rate, in gallons per second, is required to fill the tank in 20 seconds?",Easy,Medium,Easy,Medium,Easy,Medium
9,Mark works at his job for 8 hours a day for 5 days a week. He used to make 10 an hour but they raised his pay by 2 p...,Easy,Easy,Easy,Medium,Easy,Medium


### Prompt-engineering findings
1. **Label anchoring (v1 prompts):** with the zero-shot and chain-of-thought templates the 1.5B model answered
   `"Easy"` for **all 90 questions**, even while giving scores like 67. It copies the first option of the
   `"Easy|Medium|Hard"` JSON template. Accuracy collapses to chance (33%), QWK = 0.
2. **Few-shot examples** break the anchoring: the model now uses all three levels (best macro-F1 / QWK of the LLM strategies).
3. **v2 "score-first" prompt (our fix):** remove the level words entirely, give calibrated anchor questions
   (10 to 95), ask for a short reasoning plus a score, and derive the level from the score in code. This gives the
   best LLM accuracy and no label anchoring.
4. Even the best prompt is far below the **task-specific trained ensemble**. So in the final system the LLM is used
   for what it is good at, *generation and explanation*, while the trained NLP model does the scoring and
   **verifies** every LLM output.

## 2 · Live demo: rating with each strategy

In [5]:
q = 'Prove that the square root of 2 is irrational.'
for s in ['zero_shot','few_shot','chain_of_thought','score_first']:
    r = genai.rate(q, s); print(f"{s:17s} → {r['level']:6s} {r['score']:5.0f}  {r.get('reasoning','')[:150]}")

zero_shot         → Hard      67  


few_shot          → Hard      83  


chain_of_thought  → Easy      34  The question requires basic knowledge of the definition and properties of irrational numbers, specifically that square roots of non-perfect squares ar


score_first       → Medium    50  Requires understanding of rational and irrational numbers, basic algebraic manipulation, and proof by contradiction.


## 3 · Grounded explanation

In [6]:
from src.predict import get_predictor
pred = get_predictor()
r = pred.predict('Evaluate the impact of the Treaty of Versailles on the rise of fascism in Europe.')
print(r['level'], r['score']); print(genai.explain(r['english'], r))

W1116 23:29:25.588000 60899 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Medium 58.7


The exam question "Evaluate the impact of the Treaty of Versailles on the rise of fascism in Europe" is rated as Medium due to its moderate complexity level. Here are three concrete aspects that contribute to this difficulty:

1. **Bloom's Taxonomy Level (Evaluate)**: The task requires students to assess and evaluate information, which aligns with the Evaluate cognitive domain from Bloom's taxonomy. This demands higher-order thinking skills beyond recall or basic comprehension.

2. **Readability Grade (Flesch-Kincaid)**: With a readability grade of 9.1, the question is moderately complex in terms of language difficulty. It requires students to understand and interpret historical context, terminology like "Treaty of Versailles," and concepts related to fascism and its rise.

3. **Rare/Technical Words**: The use of technical words such as "fascism" and specific historical events (e.g., the Treaty of Versailles) adds complexity. These terms are not commonly encountered in everyday languag

## 4 · Controlled rewriting (verified by the trained model)

In [7]:
print('Success rate:', le['rewrite']); pd.read_csv(REPORTS/'llm_rewrite_results.csv').head(10)

Success rate: {'n': 24, 'success_rate': 0.8333333333333334, 'by_direction': {'easier': {'success': 0.833, 'mean_change': -4.567}, 'harder': {'success': 0.833, 'mean_change': 10.983}}}


,original,rewritten,direction,score_change,verified,before_level,after_level
0,"Flashlights run on batteries in order to light up. Those use chemical energy, light uses","Flashlights run on batteries in order to light up. Those use chemical energy, which is converted into light through ...",harder,31.2,True,Medium,Hard
1,"Let a b = a^b-ab . If 2 x = 22 , find x .",Let f(n) = n^2 - 3n + 10 . For what value of n is f(n) equal to 7?,harder,-0.1,False,Medium,Medium
2,Round 8899.50241201 to the nearest whole number.,"Round 8899.50241201 to the nearest whole number, considering significant figures in scientific measurements and roun...",harder,3.3,True,Hard,Hard
3,The purpose of creating Pndt d'Arc Cavern is to _ .,The purpose of creating Pndt d'Arc Cavern is to provide a unique geological feature that enhances the biodiversity o...,harder,12.4,True,Medium,Medium
4,"If you are feeling cold, its a good idea to go for a run or even a brisk walk because as you move, your energy is tu...","If you are feeling cold, it is advisable to engage in vigorous physical activity such as sprinting or cycling becaus...",harder,0.0,False,Easy,Easy
5,"If you examine eyeglasses for nearsighted people, you will find the lenses are thinnest in the center and of what sh...","Explain why the lenses in eyeglasses for nearsighted people are thinnest at the center, considering how this affects...",harder,30.0,True,Easy,Medium
6,My school's math club has 6 boys and 8 girls. I need to select a team to send to the state math competition. We want...,My school's math club has 12 boys and 16 girls. I need to select a team to send to the state math competition. We wa...,harder,10.0,True,Hard,Hard
7,The screwdriver was surprisingly sharp. This is because it's tip was what?,The screwdriver was surprisingly sharp. This is because it's tip had been honed to what precision?,harder,2.2,True,Easy,Easy
8,According to the passage you usually do everything on the first day of school except _ .,**Rewritten Question:**,harder,18.4,True,Easy,Medium
9,DNA is a vehicle for passing inherited characteristics from parent to what?,DNA is a vehicle for passing inherited characteristics from parent to offspring through which process? This process ...,harder,13.4,True,Easy,Easy


In [8]:
o = genai.rewrite('What is the chemical formula of water?', 'harder', pred)
print(json.dumps({k:o[k] for k in ['original','rewritten','before','after','score_change','verified']}, indent=1))

{
 "original": "What is the chemical formula of water?",
 "rewritten": "What is the chemical formula for water? How does its structure relate to its properties such as boiling point and solubility in polar vs non-polar substances? Discuss the implications of these properties on everyday applications like ice skating or soft drink production.",
 "before": {
  "level": "Easy",
  "score": 23.6
 },
 "after": {
  "level": "Easy",
  "score": 33.3
 },
 "score_change": 9.699999999999996,
 "verified": true
}


## 5 · Difficulty-targeted question generation

In [9]:
pd.DataFrame(genai.generate('Newton\'s laws of motion', 'Hard', 3, pred))

,question,target_level,predicted_level,predicted_score,on_target
0,Analyze the principles behind Newton's laws of motion and explain their application in spacecraft trajectory optimiz...,Hard,Medium,62.6,False
